# BANKING77: аудит и разбиение
Данные PolyAI / Casanueva et al. (2020), CC BY 4.0. Только development; test не читается. Подготовка: `python -m src.data prepare` и `python -m src.data split`.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = ROOT.parent

In [2]:
frame = pd.read_csv(ROOT / "data/processed/development.csv")
frame[["text", "category"]].info()
frame.text.str.len().describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9971 entries, 0 to 9970
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      9971 non-null   object
 1   category  9971 non-null   object
dtypes: object(2)
memory usage: 155.9+ KB


count    9971.000000
mean       59.494935
std        40.830961
min        13.000000
25%        36.000000
50%        47.000000
75%        64.000000
max       433.000000
Name: text, dtype: float64

In [3]:
audit = json.loads((ROOT / "reports/data-audit.json").read_text())
{k: v for k, v in audit.items() if k not in ["class_counts", "text_length_quantiles"]}

{'raw_rows': 10003,
 'invalid_rows': 0,
 'conflicting_rows_removed': 2,
 'normalised_duplicates_removed': 30,
 'clean_rows': 9971,
 'potential_email_rows': 0,
 'long_digit_sequence_rows': 0,
 'null_cells': 0,
 'exact_duplicate_rows': 0}

In [4]:
counts = frame.category.value_counts()
print("Class count:", len(counts), "min:", counts.min(), "max:", counts.max())
counts.to_frame("examples").head(15)

Class count: 77 min: 35 max: 187


,examples
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,172
transfer_not_received_by_recipient,171
transfer_fee_charged,171


In [5]:
parts = {
    n: pd.read_csv(ROOT / f"data/processed/{n}.csv") for n in ["train", "calibration", "validation"]
}
for a, b in [("train", "validation"), ("train", "calibration"), ("calibration", "validation")]:
    assert not set(parts[a].group) & set(parts[b].group)
pd.DataFrame({n: {"rows": len(p), "classes": p.category.nunique()} for n, p in parts.items()}).T

,rows,classes
train,5992,77
calibration,1996,77
validation,1983,77


![Development distributions](../reports/figures/eda.png)

Близкие дубликаты определяются по cosine ≥0.92 и объединяются в компоненты связности. Это не гарантия отсутствия семантических парафразов. Пользовательских и временных идентификаторов нет. Подробности: [DATA_CARD](../DATA_CARD.md).